# Day 3 Instructor Demo: tool calling and caching (D2.2, D2.6)

**Run by the instructor only** (Section 2); shared with participants afterwards. Same Secrets as the
code-along. ⚠️ Never put company data into free APIs. TinyCoder is fictional.

No LangChain here on purpose: the plain `openai` SDK shows the mechanism under every agent framework.

In [ ]:
# Setup: installs today's libraries if needed and connects to your provider
import hashlib, importlib, importlib.metadata, json, os, re, shutil, sys, time, urllib.request
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'requirements-day3.txt': 'paper-summarizer/requirements-day3.txt', 'llm_client.py': 'paper-summarizer/paper_agent/llm_client.py', 'paper_agent/__init__.py': 'paper-summarizer/paper_agent/__init__.py', 'paper_agent/rag/__init__.py': 'paper-summarizer/paper_agent/rag/__init__.py', 'paper_agent/rag/pdf_rag.py': 'paper-summarizer/paper_agent/rag/pdf_rag.py', 'paper_agent/agent/__init__.py': 'paper-summarizer/paper_agent/agent/__init__.py', 'paper_agent/agent/policy.py': 'paper-summarizer/paper_agent/agent/policy.py', 'paper_agent/agent/fakes.py': 'paper-summarizer/paper_agent/agent/fakes.py', 'paper_agent/agent/trace.py': 'paper-summarizer/paper_agent/agent/trace.py', 'tinycoder_day3.pdf': 'paper-summarizer/paper_agent/fixtures/pdfs/tinycoder_day3.pdf', 'tinycoder_injected.pdf': 'paper-summarizer/paper_agent/fixtures/pdfs/tinycoder_injected.pdf', 'second_paper.pdf': 'paper-summarizer/paper_agent/fixtures/pdfs/second_paper.pdf', 'tinycoder_embeddings_day3.mock-embed.json': 'notebooks/day3/tinycoder_embeddings_day3.mock-embed.json'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    Path(name).parent.mkdir(parents=True, exist_ok=True)
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)

fetch("requirements-day3.txt", force=True)
PINS = dict(line.split("==") for line in Path("requirements-day3.txt").read_text().split() if "==" in line)

def installed(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

wrong = {p: installed(p) for p, v in PINS.items() if installed(p) != v}
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if wrong and not IN_COLAB:                                    # never pip-install into someone's own Python
    raise RuntimeError(f"Not in Colab, and these differ from requirements-day3.txt: {wrong}. "
                       "Install them in this environment yourself (pip install -r requirements-day3.txt).")
if wrong:
    already_loaded = [m for m in ("langchain", "langchain_core", "langgraph", "langchain_openai") if m in sys.modules]
    print("Installing LangChain and today's libraries (pinned): a few minutes the first time ...")
    !pip install -q -r requirements-day3.txt
    if already_loaded:
        print("\n" + "=" * 72 + "\n✅ Installed. ⚠️  Now RESTART THE SESSION (Runtime → Restart session), then run 3.0 again.\n"
              "An older LangChain was already loaded in this runtime; the new one is used only after a restart.\n" + "=" * 72)
        raise SystemExit("restart the session, then run 3.0")

NAMES = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")
try:
    from google.colab import userdata
    for name in NAMES:
        try:
            if userdata.get(name):
                os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass
from openai import OpenAI
client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"], max_retries=2, timeout=30)
MODEL = os.environ["LLM_MODEL"].split(",")[0].strip()
print("✅ ready:", MODEL)

---
## D2.2 How tool calling works · *Slide 2.2*

One request with a `tools` list. The reply has no text: just a tool name and arguments. Nothing has run.

In [ ]:
# D2.2: the raw request
tools = [{"type": "function", "function": {
    "name": "search_paper",
    "description": "Search the current paper for the passages most relevant to a question. "
                   "Use it for anything about what the paper says, measures or claims.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string"},
                                                    "k": {"type": "integer", "default": 4}},
                   "required": ["query"]}}}]
request = {"model": MODEL, "temperature": 0, "tools": tools,
           "messages": [{"role": "user", "content": "How does TinyCoder do on long files?"}]}
print(json.dumps(request, indent=2))

In [ ]:
# D2.2: the raw reply
response = client.chat.completions.create(**request)
print(json.dumps(response.model_dump(exclude_none=True), indent=2))
call = response.choices[0].message.tool_calls[0]
print(f"\nThe model asks for {call.function.name}({call.function.arguments}). Our code runs it and sends the "
      "result back as a 'tool' message.")

---
## D2.6 Four places to cache · *Slide 2.6*

An exact-match cache in five lines: the key is the whole request. The same call twice, then a one-word change.

In [ ]:
# D2.6
import hashlib
CACHE = {}

def cached_call(question):
    body = {"model": MODEL, "temperature": 0, "max_tokens": 200,
            "messages": [{"role": "user", "content": question}]}
    key = hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()   # the whole request
    start = time.perf_counter()
    hit = key in CACHE
    if not hit:
        CACHE[key] = client.chat.completions.create(**body)
    return CACHE[key], hit, (time.perf_counter() - start) * 1000

for question in ("What is sliding-window attention?", "What is sliding-window attention?",
                 "What is sliding-window attention exactly?"):
    reply, hit, ms = cached_call(question)
    print(f"{'HIT ' if hit else 'MISS'} {ms:8.1f} ms  {question!r}")